# Using the ExcelDateToDateTime Module in baseobjects

## Introduction

The `exceldatetodatetime` module provides functionality for converting Excel date values to Python datetime objects. Excel stores dates as floating-point numbers, where the integer part represents the number of days since a reference date (December 30, 1899), and the fractional part represents the time of day.

This module contains a single function, `excel_date_to_datetime`, which handles the conversion from Excel date values to Python datetime objects. The function supports multiple input types (int, float, str, bytes) and allows you to specify the timezone for the resulting datetime object.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `exceldatetodatetime` module
- Using the `excel_date_to_datetime` function with different input types
- Working with timezone information
- Practical examples and use cases

**Prerequisites:**
- Basic understanding of Python's datetime module
- Familiarity with Excel's date representation
- Knowledge of timezone concepts

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [12]:
from baseobjects.operations.exceldatetodatetime import excel_date_to_datetime
from datetime import datetime, timezone, timedelta

## Core Functionality

The `excel_date_to_datetime` function converts Excel date values to Python datetime objects. Let's explore its basic functionality.

### Basic Usage

Let's start with a simple example to see how the function works:

In [13]:
# Convert an Excel date to a datetime object
excel_date = 44000  # Approximately March 6, 2020
dt = excel_date_to_datetime(excel_date)
print(f"Excel date: {excel_date}")
print(f"Python datetime: {dt}")
print(f"Year: {dt.year}, Month: {dt.month}, Day: {dt.day}")

Excel date: 44000
Python datetime: 2020-06-18 00:00:00+00:00
Year: 2020, Month: 6, Day: 18


By default, the function returns a datetime object with UTC timezone. Let's try with a date that includes a time component:

In [14]:
# Convert an Excel date with time component
excel_date_with_time = 44000.5  # March 6, 2020, 12:00 PM
dt = excel_date_to_datetime(excel_date_with_time)
print(f"Excel date with time: {excel_date_with_time}")
print(f"Python datetime: {dt}")
print(f"Year: {dt.year}, Month: {dt.month}, Day: {dt.day}")
print(f"Hour: {dt.hour}, Minute: {dt.minute}, Second: {dt.second}")

Excel date with time: 44000.5
Python datetime: 2020-06-18 12:00:00+00:00
Year: 2020, Month: 6, Day: 18
Hour: 12, Minute: 0, Second: 0


### Different Input Types

The `excel_date_to_datetime` function supports multiple input types. Let's see how it handles different types:

In [15]:
# Integer input
int_date = 44000
dt_from_int = excel_date_to_datetime(int_date)
print(f"From integer: {dt_from_int}")

# Float input
float_date = 44000.75  # With time component (18:00)
dt_from_float = excel_date_to_datetime(float_date)
print(f"From float: {dt_from_float}")

# String input
string_date = "44000.5"  # With time component (12:00)
dt_from_string = excel_date_to_datetime(string_date)
print(f"From string: {dt_from_string}")

# Bytes input
bytes_date = b"44000.25"  # With time component (06:00)
dt_from_bytes = excel_date_to_datetime(bytes_date)
print(f"From bytes: {dt_from_bytes}")

From integer: 2020-06-18 00:00:00+00:00
From float: 2020-06-18 18:00:00+00:00
From string: 2020-06-18 12:00:00+00:00
From bytes: 2020-06-18 06:00:00+00:00


### Working with Timezones

The `excel_date_to_datetime` function allows you to specify the timezone for the resulting datetime object through the `tzinfo` parameter:

In [16]:
# Default timezone (UTC)
dt_utc = excel_date_to_datetime(44000)
print(f"Default timezone (UTC): {dt_utc}")

# Eastern Time (UTC-5)
eastern = timezone(timedelta(hours=-5))
dt_eastern = excel_date_to_datetime(44000, tzinfo=eastern)
print(f"Eastern Time (UTC-5): {dt_eastern}")

# Pacific Time (UTC-8)
pacific = timezone(timedelta(hours=-8))
dt_pacific = excel_date_to_datetime(44000, tzinfo=pacific)
print(f"Pacific Time (UTC-8): {dt_pacific}")

# No timezone (None)
dt_none = excel_date_to_datetime(44000, tzinfo=None)
print(f"No timezone: {dt_none}")

Default timezone (UTC): 2020-06-18 00:00:00+00:00
Eastern Time (UTC-5): 2020-06-18 00:00:00-05:00
Pacific Time (UTC-8): 2020-06-18 00:00:00-08:00
No timezone: 2020-06-18 00:00:00


## Advanced Features

### Understanding Excel's Date System

Excel has two date systems:
1. The 1900 date system (default in Windows): January 1, 1900 is represented as 1
2. The 1904 date system (default in Mac Excel before 2016): January 1, 1904 is represented as 0

The `exceldatetodatetime` module uses the 1900 date system, where December 30, 1899 is represented as 0. Let's explore this:

In [17]:
# Excel's reference date (day 0)
reference_date = excel_date_to_datetime(0)
print(f"Excel reference date (day 0): {reference_date}")

# Day 1 in Excel
day_1 = excel_date_to_datetime(1)
print(f"Day 1 in Excel: {day_1}")

# Day 60 (Note: Excel has a leap year bug where it incorrectly treats 1900 as a leap year)
day_60 = excel_date_to_datetime(60)
print(f"Day 60 in Excel: {day_60}")

# Negative days (before the reference date)
day_minus_10 = excel_date_to_datetime(-10)
print(f"10 days before reference date: {day_minus_10}")

Excel reference date (day 0): 1899-12-30 00:00:00+00:00
Day 1 in Excel: 1899-12-31 00:00:00+00:00
Day 60 in Excel: 1900-02-28 00:00:00+00:00
10 days before reference date: 1899-12-20 00:00:00+00:00


### Converting Between Excel Dates and Python Datetimes

We can also convert from Python datetime objects to Excel dates (although this is not directly provided by the module):

In [18]:
def datetime_to_excel_date(dt):
    """Convert a Python datetime to an Excel date value."""
    # Excel's reference date
    excel_ref = datetime(1899, 12, 30, tzinfo=dt.tzinfo if dt.tzinfo else None)
    
    # Calculate the difference in days
    delta = dt - excel_ref
    
    # Return the total days (including fractional part for time)
    return delta.total_seconds() / (24 * 60 * 60)

# Current date and time
now = datetime.now(timezone.utc)
excel_now = datetime_to_excel_date(now)
print(f"Current datetime: {now}")
print(f"As Excel date: {excel_now}")

# Convert back to datetime
now_roundtrip = excel_date_to_datetime(excel_now)
print(f"Converted back to datetime: {now_roundtrip}")
print(f"Difference in seconds: {(now - now_roundtrip).total_seconds()}")

Current datetime: 2025-07-25 18:12:11.823662+00:00
As Excel date: 45863.758470181274
Converted back to datetime: 2025-07-25 18:12:11.823662+00:00
Difference in seconds: 0.0


## Examples

Let's explore some practical examples of using the `excel_date_to_datetime` function.

### Example 1: Processing Excel Data

Imagine you're working with data exported from Excel and need to convert date values to Python datetime objects:

In [19]:
# Sample data exported from Excel (column of dates)
excel_dates = [
    44197,       # January 19, 2021
    44197.25,    # January 19, 2021, 06:00
    44197.5,     # January 19, 2021, 12:00
    44197.75,    # January 19, 2021, 18:00
    44198        # January 20, 2021
]

# Convert all dates to datetime objects
python_dates = [excel_date_to_datetime(date) for date in excel_dates]

# Display the results
for excel_date, python_date in zip(excel_dates, python_dates):
    print(f"Excel: {excel_date} -> Python: {python_date}")

Excel: 44197 -> Python: 2021-01-01 00:00:00+00:00
Excel: 44197.25 -> Python: 2021-01-01 06:00:00+00:00
Excel: 44197.5 -> Python: 2021-01-01 12:00:00+00:00
Excel: 44197.75 -> Python: 2021-01-01 18:00:00+00:00
Excel: 44198 -> Python: 2021-01-02 00:00:00+00:00


### Example 2: Working with Different Timezones

If you're working with dates from different timezones, you can specify the appropriate timezone for each date:

In [20]:
# Define some timezones
utc = timezone.utc
eastern = timezone(timedelta(hours=-5))
pacific = timezone(timedelta(hours=-8))
central_europe = timezone(timedelta(hours=1))
japan = timezone(timedelta(hours=9))

# Sample meeting times in Excel format (same day, different timezones)
meeting_date = 44197.5  # January 19, 2021, 12:00 UTC

# Convert to different timezones
meetings = {
    "UTC": excel_date_to_datetime(meeting_date, tzinfo=utc),
    "Eastern": excel_date_to_datetime(meeting_date, tzinfo=eastern),
    "Pacific": excel_date_to_datetime(meeting_date, tzinfo=pacific),
    "Central Europe": excel_date_to_datetime(meeting_date, tzinfo=central_europe),
    "Japan": excel_date_to_datetime(meeting_date, tzinfo=japan)
}

# Display meeting times in different timezones
print("Meeting times in different timezones:")
for zone, time in meetings.items():
    print(f"{zone}: {time}")

Meeting times in different timezones:
UTC: 2021-01-01 12:00:00+00:00
Eastern: 2021-01-01 12:00:00-05:00
Pacific: 2021-01-01 12:00:00-08:00
Central Europe: 2021-01-01 12:00:00+01:00
Japan: 2021-01-01 12:00:00+09:00


### Example 3: Handling Date Ranges

You can use the `excel_date_to_datetime` function to work with date ranges:

In [21]:
# Generate a range of dates
start_date = 44197  # January 19, 2021
end_date = 44204    # January 26, 2021

# Create a list of dates in the range
date_range = [start_date + i for i in range(end_date - start_date + 1)]

# Convert to datetime objects
datetime_range = [excel_date_to_datetime(date) for date in date_range]

# Display the date range
print("Date range:")
for date in datetime_range:
    print(f"{date.strftime('%A, %B %d, %Y')}")

Date range:
Friday, January 01, 2021
Saturday, January 02, 2021
Sunday, January 03, 2021
Monday, January 04, 2021
Tuesday, January 05, 2021
Wednesday, January 06, 2021
Thursday, January 07, 2021
Friday, January 08, 2021


## API Highlights

The `exceldatetodatetime` module provides a single function:

### excel_date_to_datetime

```python
@singlekwargdispatch
def excel_date_to_datetime(timestamp: int | float | str | bytes, tzinfo: TZInfo | None = timezone.utc) -> datetime:
    """Converts a filetime to a datetime object.

    Args:
        timestamp: The filetime to convert to a datetime.
        tzinfo: The timezone of the datetime.

    Returns:
        The datetime of the filetime.
    """
```

Parameters:
- `timestamp`: The Excel date value to convert. Can be an int, float, str, or bytes.
- `tzinfo`: The timezone to use for the resulting datetime object. Defaults to UTC.

Returns:
- A datetime object representing the Excel date.

The function uses the `singlekwargdispatch` decorator to handle different input types, with specialized implementations for:
- int and float
- str and bytes

## Troubleshooting / FAQs

### Q: Why does my Excel date convert to a different date than expected?

A: There are a few possible reasons:
1. Excel has a leap year bug where it incorrectly treats 1900 as a leap year. This means that dates after February 28, 1900, will be off by one day.
2. The timezone might be different. By default, the function returns a datetime with UTC timezone.
3. Excel has two different date systems (1900 and 1904). This module uses the 1900 date system.

### Q: How do I handle dates before the Excel reference date (December 30, 1899)?

A: You can use negative numbers to represent dates before the reference date. For example, -1 represents December 29, 1899.

### Q: Why does the function accept string and bytes inputs?

A: This provides flexibility when working with data from different sources. For example, if you're reading data from a file or a network stream, you might get the Excel date as a string or bytes.

### Q: How accurate is the conversion?

A: The conversion is accurate to the microsecond level, which is the precision of Python's datetime objects. Excel dates have a precision of 1/86400 (one second), so there should be no loss of precision in the conversion.

## Conclusion and Next Steps

In this tutorial, we've explored the `exceldatetodatetime` module and its `excel_date_to_datetime` function. We've seen how to convert Excel date values to Python datetime objects, how to work with different input types and timezones, and how to use the function in practical examples.

The `excel_date_to_datetime` function is a useful utility for working with data exported from Excel or for interoperating with systems that use Excel's date representation. It handles the conversion seamlessly, allowing you to work with dates in the more flexible and powerful datetime format.

### Next Steps

- Explore other modules in the baseobjects.operations package, such as `filetimetodatetime` for working with Windows FILETIME values.
- Check out the datetime module in the Python standard library for more functionality related to date and time manipulation.
- Consider how you might use `excel_date_to_datetime` in your own projects that involve data from Excel or other systems that use similar date representations.